In [1]:
import pandas as pd
from pathlib import Path
from pandasql import sqldf
import sqlite3
import numpy as np
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib
import os

In [2]:
db_path = Path(r'C:\Users\Art\Documents\civ5\Civ5DebugDatabase.db')

In [3]:
cnx = sqlite3.connect(db_path)

In [4]:
pd.read_sql_query('select * from features', cnx)

,ID,Type,Description,Civilopedia,Help,ArtDefineTag,StartingLocationWeight,Movement,SeeThrough,Defense,...,AdvancedStartRemoveCost,PortraitIndex,IconAtlas,PassableTechFeature,FreePromotionIfOwned,LocationUnitFreePromotion,SpawnLocationUnitFreePromotion,AdjacentSpawnLocationUnitFreePromotion,PseudoNaturalWonder,ExtraTurnDamage
0,0,FEATURE_ICE,TXT_KEY_FEATURE_ICE,TXT_KEY_FEATURE_ICE_PEDIA,None,ART_DEF_FEATURE_ICE,0,1,0,0,...,0,6,TERRAIN_ATLAS,None,None,None,None,None,0,0
1,1,FEATURE_JUNGLE,TXT_KEY_FEATURE_JUNGLE,TXT_KEY_CIV5_FEATURES_JUNGLE_TEXT,None,ART_DEF_FEATURE_JUNGLE,-10,2,1,25,...,0,7,TERRAIN_ATLAS,None,None,None,PROMOTION_WOODS_WALKER,None,0,0
2,2,FEATURE_MARSH,TXT_KEY_FEATURE_MARSH,TXT_KEY_CIV5_FEATURES_MARSH_TEXT,None,ART_DEF_FEATURE_MARSH,-10,3,0,5,...,0,8,TERRAIN_ATLAS,None,None,None,None,None,0,0
3,3,FEATURE_OASIS,TXT_KEY_FEATURE_OASIS,TXT_KEY_CIV5_FEATURES_OASIS_TEXT,None,ART_DEF_FEATURE_OASIS,0,1,0,5,...,0,0,NEW_TERRAIN_ATLAS_DLC,None,None,None,None,None,0,0
4,4,FEATURE_FLOOD_PLAINS,TXT_KEY_FEATURE_FLOOD_PLAINS,TXT_KEY_CIV5_FEATURES_FLOODPLAINS_TEXT,None,ART_DEF_FEATURE_FLOOD_PLAINS,0,1,0,5,...,0,2,TERRAIN_ATLAS,None,None,None,None,None,0,0
5,5,FEATURE_FOREST,TXT_KEY_FEATURE_FOREST,TXT_KEY_CIV5_FEATURES_FOREST_TEXT,None,ART_DEF_FEATURE_FOREST,0,2,1,25,...,0,3,TERRAIN_ATLAS,None,None,None,PROMOTION_WOODS_WALKER,None,0,0
6,6,FEATURE_FALLOUT,TXT_KEY_FEATURE_FALLOUT,TXT_KEY_FEATURE_FALLOUT_PEDIA,None,ART_DEF_FEATURE_FALLOUT,0,2,0,5,...,0,17,TERRAIN_ATLAS,None,None,None,None,None,0,-10
7,7,FEATURE_CRATER,TXT_KEY_FEATURE_CRATER,TXT_KEY_CIV5_FEATURES_BARRINGER_TEXT,None,ART_DEF_FEATURE_CRATER,0,2,2,10,...,0,2,NW_ATLAS,None,None,None,None,None,0,0
8,8,FEATURE_FUJI,TXT_KEY_FEATURE_MT_FUJI,TXT_KEY_CIV5_FEATURES_FUJI_TEXT,None,ART_DEF_FEATURE_FUJI,0,1,2,0,...,0,3,NW_ATLAS,None,None,None,None,None,0,0
9,9,FEATURE_MESA,TXT_KEY_FEATURE_MESA,TXT_KEY_CIV5_FEATURES_GRANDMESA_TEXT,None,ART_DEF_FEATURE_MESA,0,2,2,10,...,0,7,NW_ATLAS,None,None,None,None,None,0,0


In [5]:
CIV_TAG_TO_TEXT_MAP = {
    "CIVILIZATION_AMERICA": "America",
    "CIVILIZATION_ARABIA": "Arabia",
    "CIVILIZATION_AZTEC": "The Aztecs",
    "CIVILIZATION_CHINA": "China",
    "CIVILIZATION_EGYPT": "Egypt",
    "CIVILIZATION_ENGLAND": "England",
    "CIVILIZATION_FRANCE": "France",
    "CIVILIZATION_GERMANY": "Germany",
    "CIVILIZATION_GREECE": "Greece",
    "CIVILIZATION_INDIA": "India",
    "CIVILIZATION_IROQUOIS": "The Iroquois",
    "CIVILIZATION_JAPAN": "Japan",
    "CIVILIZATION_OTTOMAN": "The Ottomans",
    "CIVILIZATION_PERSIA": "Persia",
    "CIVILIZATION_ROME": "Rome",
    "CIVILIZATION_RUSSIA": "Russia",
    "CIVILIZATION_SIAM": "Siam",
    "CIVILIZATION_SONGHAI": "Songhai",
    "CIVILIZATION_MINOR": "City State",
    "CIVILIZATION_BARBARIAN": "Barbarians",
    "CIVILIZATION_MONGOL": "Mongolia",
    "CIVILIZATION_INCA": "The Inca",
    "CIVILIZATION_SPAIN": "Spain",
    "CIVILIZATION_POLYNESIA": "Polynesia",
    "CIVILIZATION_DENMARK": "Denmark",
    "CIVILIZATION_KOREA": "Korea",
    "CIVILIZATION_BABYLON": "Babylon",
    "CIVILIZATION_AUSTRIA": "Austria",
    "CIVILIZATION_BYZANTIUM": "Byzantium",
    "CIVILIZATION_CARTHAGE": "Carthage",
    "CIVILIZATION_CELTS": "The Celts",
    "CIVILIZATION_ETHIOPIA": "Ethiopia",
    "CIVILIZATION_HUNS": "The Huns",
    "CIVILIZATION_MAYA": "The Maya",
    "CIVILIZATION_NETHERLANDS": "The Netherlands",
    "CIVILIZATION_SWEDEN": "Sweden",
    "CIVILIZATION_ASSYRIA": "Assyria",
    "CIVILIZATION_BRAZIL": "Brazil",
    "CIVILIZATION_INDONESIA": "Indonesia",
    "CIVILIZATION_MOROCCO": "Morocco",
    "CIVILIZATION_POLAND": "Poland",
    "CIVILIZATION_PORTUGAL": "Portugal",
    "CIVILIZATION_SHOSHONE": "The Shoshone",
    "CIVILIZATION_VENICE": "Venice",
    "CIVILIZATION_ZULU": "The Zulus"
}

In [6]:
civ_flavors_df = pd.read_sql_query('''
SELECT
    CivilizationType,
    FlavorType,
    Flavor
FROM Leader_Flavors, Civilization_Leaders
WHERE Leader_Flavors.LeaderType = Civilization_Leaders.LeaderheadType
AND CivilizationType != "CIVILIZATION_MINOR"
AND CivilizationType != "CIVILIZATION_BARBARIAN"

UNION

SELECT
    CivilizationType,
    MajorCivApproachType AS FlavorType,
    Bias AS Flavor
FROM Leader_MajorCivApproachBiases, Civilization_Leaders
WHERE Leader_MajorCivApproachBiases.LeaderType = Civilization_Leaders.LeaderheadType
AND CivilizationType != "CIVILIZATION_MINOR"
AND CivilizationType != "CIVILIZATION_BARBARIAN"
''', cnx)

civ_flavors_df['civ'] = civ_flavors_df['CivilizationType'].map(CIV_TAG_TO_TEXT_MAP)
civ_flavors_df = sqldf('''
SELECT
  civ,
  FlavorType,
  Flavor
FROM civ_flavors_df
GROUP BY civ, FlavorType
''')
civ_flavors_df

,civ,FlavorType,Flavor
0,America,FLAVOR_AIR,5
1,America,FLAVOR_AIRLIFT,5
2,America,FLAVOR_AIR_CARRIER,5
3,America,FLAVOR_ANTIAIR,7
4,America,FLAVOR_ARCHAEOLOGY,9
...,...,...,...
1930,Venice,MAJOR_CIV_APPROACH_FRIENDLY,8
1931,Venice,MAJOR_CIV_APPROACH_GUARDED,5
1932,Venice,MAJOR_CIV_APPROACH_HOSTILE,3
1933,Venice,MAJOR_CIV_APPROACH_NEUTRAL,5


In [7]:
leader_personality_df = pd.read_sql_query('''
SELECT
    Type,
    CivilizationType,
    Description,
    Personality,
    PrimaryVictoryPursuit,
    SecondaryVictoryPursuit
FROM Leaders, Civilization_Leaders
WHERE Leaders.Type = Civilization_Leaders.LeaderheadType
AND CivilizationType != "CIVILIZATION_MINOR"
AND CivilizationType != "CIVILIZATION_BARBARIAN"
''', cnx)

leader_personality_df['civ'] = leader_personality_df['CivilizationType'].map(CIV_TAG_TO_TEXT_MAP)
leader_personality_df['Personality'] = leader_personality_df['Personality'].str.replace('PERSONALITY_', '')
leader_personality_df['PrimaryVictoryPursuit'] = leader_personality_df['PrimaryVictoryPursuit'].str.replace('VICTORY_PURSUIT_', '')
leader_personality_df['SecondaryVictoryPursuit'] = leader_personality_df['SecondaryVictoryPursuit'].str.replace('VICTORY_PURSUIT_', '')
leader_personality_df

,Type,CivilizationType,Description,Personality,PrimaryVictoryPursuit,SecondaryVictoryPursuit,civ
0,LEADER_WASHINGTON,CIVILIZATION_AMERICA,TXT_KEY_LEADER_WASHINGTON,COALITION,None,None,America
1,LEADER_HARUN_AL_RASHID,CIVILIZATION_ARABIA,TXT_KEY_LEADER_HARUN,COALITION,CULTURE,None,Arabia
2,LEADER_MONTEZUMA,CIVILIZATION_AZTEC,TXT_KEY_LEADER_MONTEZUMA,CONQUEROR,DOMINATION,None,The Aztecs
3,LEADER_WU_ZETIAN,CIVILIZATION_CHINA,TXT_KEY_LEADER_WU_ZETIAN,EXPANSIONIST,SCIENCE,DOMINATION,China
4,LEADER_RAMESSES,CIVILIZATION_EGYPT,TXT_KEY_LEADER_RAMESSES,COALITION,CULTURE,None,Egypt
5,LEADER_ELIZABETH,CIVILIZATION_ENGLAND,TXT_KEY_LEADER_ELIZABETH,COALITION,DOMINATION,SCIENCE,England
6,LEADER_NAPOLEON,CIVILIZATION_FRANCE,TXT_KEY_LEADER_NAPOLEON,CONQUEROR,DOMINATION,CULTURE,France
7,LEADER_BISMARCK,CIVILIZATION_GERMANY,TXT_KEY_LEADER_BISMARCK,DIPLOMAT,DIPLOMACY,None,Germany
8,LEADER_ALEXANDER,CIVILIZATION_GREECE,TXT_KEY_LEADER_ALEXANDER,EXPANSIONIST,DIPLOMACY,DOMINATION,Greece
9,LEADER_GANDHI,CIVILIZATION_INDIA,TXT_KEY_LEADER_GANDHI,DIPLOMAT,SCIENCE,CULTURE,India


In [8]:
pd.pivot_table(civ_flavors_df, index=['civ'], columns=['FlavorType'], values='Flavor').merge(
    leader_personality_df,
    on='civ',
    how='left'
).to_csv('civ_flavors.csv')

In [9]:
# pd.pivot_table(civ_flavors_df, index=['civ'], columns=['FlavorType'], values='Flavor').to_csv('civ_flavors.csv')

In [10]:
civ_colors_df = pd.read_sql_query('''
SELECT
    Civilizations.Type,
    Colors.Red * 255 AS red,
    Colors.Green * 255 AS green,
    Colors.Blue * 255 AS blue
FROM
    Civilizations, PlayerColors, Colors
WHERE
    Civilizations.DefaultPlayerColor = PlayerColors.Type AND
    PlayerColors.PrimaryColor = Colors.Type
''', cnx)


civ_colors_df['civ'] = civ_colors_df['Type'].map(CIV_TAG_TO_TEXT_MAP)
civ_colors_df = civ_colors_df.drop(columns=['Type']).round()
civ_colors_df

,red,green,blue,civ
0,255.0,255.0,255.0,America
1,146.0,221.0,10.0,Arabia
2,137.0,239.0,213.0,The Aztecs
3,255.0,255.0,255.0,China
4,83.0,0.0,208.0,Egypt
5,255.0,255.0,255.0,England
6,235.0,235.0,139.0,France
7,37.0,43.0,33.0,Germany
8,65.0,141.0,254.0,Greece
9,255.0,153.0,50.0,India


In [11]:
civ_colors_df.to_csv('civ_colors.csv')

In [12]:
civ_bg_colors_df = pd.read_sql_query('''
SELECT
    Civilizations.Type,
    Colors.Red * 255 AS red,
    Colors.Green * 255 AS green,
    Colors.Blue * 255 AS blue
FROM
    Civilizations, PlayerColors, Colors
WHERE
    Civilizations.DefaultPlayerColor = PlayerColors.Type AND
    PlayerColors.SecondaryColor = Colors.Type
''', cnx)


civ_bg_colors_df['civ'] = civ_bg_colors_df['Type'].map(CIV_TAG_TO_TEXT_MAP)
civ_bg_colors_df = civ_bg_colors_df.drop(columns=['Type'])
civ_bg_colors_df

,red,green,blue,civ
0,31.1100,51.0000,120.10500,America
1,43.0950,87.9750,45.90000,Arabia
2,161.4150,57.1200,34.93500,The Aztecs
3,0.0000,148.9200,82.11000,China
4,255.0000,251.9400,3.06000,Egypt
5,108.8850,2.0400,0.00000,England
6,65.0250,141.0150,253.98000,France
7,179.0100,177.9900,184.11000,Germany
8,255.0000,255.0000,255.00000,Greece
9,18.1050,135.9150,6.88500,India


In [13]:
civ_bg_colors_df.to_csv('civ_bg_colors.csv')

In [14]:
pd.read_sql_query("SELECT * FROM PlayerColors", cnx)

,ID,Type,PrimaryColor,SecondaryColor,TextColor
0,0,PLAYERCOLOR_BLACK,COLOR_PLAYER_BLACK,COLOR_PLAYER_WHITE,COLOR_PLAYER_BLACK_TEXT
1,1,PLAYERCOLOR_BLUE,COLOR_PLAYER_BLUE,COLOR_PLAYER_WHITE,COLOR_PLAYER_BLUE_TEXT
2,2,PLAYERCOLOR_BROWN,COLOR_PLAYER_BROWN,COLOR_PLAYER_DARK_YELLOW,COLOR_PLAYER_BROWN_TEXT
3,3,PLAYERCOLOR_CYAN,COLOR_PLAYER_CYAN,COLOR_PLAYER_BLACK,COLOR_PLAYER_CYAN_TEXT
4,4,PLAYERCOLOR_DARK_BLUE,COLOR_PLAYER_DARK_BLUE,COLOR_PLAYER_YELLOW,COLOR_PLAYER_DARK_BLUE_TEXT
...,...,...,...,...,...
101,101,PLAYERCOLOR_POLAND,COLOR_PLAYER_POLAND_ICON,COLOR_PLAYER_POLAND_BACKGROUND,COLOR_PLAYER_WHITE_TEXT
102,102,PLAYERCOLOR_PORTUGAL,COLOR_PLAYER_PORTUGAL_ICON,COLOR_PLAYER_PORTUGAL_BACKGROUND,COLOR_PLAYER_WHITE_TEXT
103,103,PLAYERCOLOR_SHOSHONE,COLOR_PLAYER_SHOSHONE_ICON,COLOR_PLAYER_SHOSHONE_BACKGROUND,COLOR_PLAYER_WHITE_TEXT
104,104,PLAYERCOLOR_VENICE,COLOR_PLAYER_VENICE_ICON,COLOR_PLAYER_VENICE_BACKGROUND,COLOR_PLAYER_WHITE_TEXT


In [15]:
pd.read_sql_query("SELECT * FROM Colors", cnx)

,ID,Type,Red,Green,Blue,Alpha
0,0,COLOR_CLEAR,1.000000,1.000000,1.000000,0.00
1,1,COLOR_ALPHA_GREY,0.100000,0.100000,0.100000,0.45
2,2,COLOR_WHITE,1.000000,1.000000,1.000000,1.00
3,3,COLOR_BLACK,0.000000,0.000000,0.000000,1.00
4,4,COLOR_DARK_GREY,0.250000,0.250000,0.250000,1.00
...,...,...,...,...,...,...
215,217,COLOR_PLAYER_SHOSHONE_BACKGROUND,0.290196,0.231373,0.180392,1.00
216,218,COLOR_PLAYER_VENICE_ICON,1.000000,0.996078,0.847059,1.00
217,219,COLOR_PLAYER_VENICE_BACKGROUND,0.400000,0.133333,0.635294,1.00
218,220,COLOR_PLAYER_ZULU_ICON,0.419608,0.196078,0.098039,1.00


In [16]:
pd.read_sql_query("SELECT * FROM Civilizations", cnx)

,ID,Type,Description,Civilopedia,CivilopediaTag,Strategy,Playable,AIPlayable,ShortDescription,Adjective,...,DerivativeCiv,PortraitIndex,IconAtlas,AlphaIconAtlas,MapImage,DawnOfManQuote,DawnOfManImage,DawnOfManAudio,PackageID,SoundtrackTag
0,0,CIVILIZATION_AMERICA,TXT_KEY_CIV_AMERICA_DESC,None,TXT_KEY_CIV5_UNITEDSTATES,None,1,1,TXT_KEY_CIV_AMERICA_SHORT_DESC,TXT_KEY_CIV_AMERICA_ADJECTIVE,...,None,0,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapAmerica512.dds,TXT_KEY_CIV5_DAWN_UNITEDSTATES_TEXT,DOM_Washington.dds,AS2D_DOM_SPEECH_UNITED_STATES,None,None
1,1,CIVILIZATION_ARABIA,TXT_KEY_CIV_ARABIA_DESC,None,TXT_KEY_CIV5_ARABIA,None,1,1,TXT_KEY_CIV_ARABIA_SHORT_DESC,TXT_KEY_CIV_ARABIA_ADJECTIVE,...,None,1,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapAbbasid512.dds,TXT_KEY_CIV5_DAWN_ARABIA_TEXT,DOM_AlRashid.dds,AS2D_DOM_SPEECH_ARABIA,None,None
2,2,CIVILIZATION_AZTEC,TXT_KEY_CIV_AZTEC_DESC,None,TXT_KEY_CIV5_AZTECS,None,1,1,TXT_KEY_CIV_AZTEC_SHORT_DESC,TXT_KEY_CIV_AZTEC_ADJECTIVE,...,None,2,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapAztec512.dds,TXT_KEY_CIV5_DAWN_AZTECS_TEXT,DOM_Monty.dds,AS2D_DOM_SPEECH_AZTEC,None,None
3,3,CIVILIZATION_CHINA,TXT_KEY_CIV_CHINA_DESC,None,TXT_KEY_CIV5_CHINA,None,1,1,TXT_KEY_CIV_CHINA_SHORT_DESC,TXT_KEY_CIV_CHINA_ADJECTIVE,...,None,4,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapChina512.dds,TXT_KEY_CIV5_DAWN_CHINA_TEXT,DOM_Wu.dds,AS2D_DOM_SPEECH_CHINA,None,None
4,4,CIVILIZATION_EGYPT,TXT_KEY_CIV_EGYPT_DESC,None,TXT_KEY_CIV5_EGYPT,None,1,1,TXT_KEY_CIV_EGYPT_SHORT_DESC,TXT_KEY_CIV_EGYPT_ADJECTIVE,...,None,5,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapEgypt512.dds,TXT_KEY_CIV5_DAWN_EGYPT_TEXT,DOM_Ramesess.dds,AS2D_DOM_SPEECH_EGYPT,None,None
5,5,CIVILIZATION_ENGLAND,TXT_KEY_CIV_ENGLAND_DESC,None,TXT_KEY_CIV5_ENGLAND,None,1,1,TXT_KEY_CIV_ENGLAND_SHORT_DESC,TXT_KEY_CIV_ENGLAND_ADJECTIVE,...,None,6,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapEngland512.dds,TXT_KEY_CIV5_DAWN_ENGLAND_TEXT,DOM_Elizabeth.dds,AS2D_DOM_SPEECH_ENGLAND,None,None
6,6,CIVILIZATION_FRANCE,TXT_KEY_CIV_FRANCE_DESC,None,TXT_KEY_CIV5_FRANCE,None,1,1,TXT_KEY_CIV_FRANCE_SHORT_DESC,TXT_KEY_CIV_FRANCE_ADJECTIVE,...,None,7,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapFrance512.dds,TXT_KEY_CIV5_DAWN_FRANCE_TEXT,DOM_Napoleon.dds,AS2D_DOM_SPEECH_FRANCE,None,None
7,7,CIVILIZATION_GERMANY,TXT_KEY_CIV_GERMANY_DESC,None,TXT_KEY_CIV5_GERMANY,None,1,1,TXT_KEY_CIV_GERMANY_SHORT_DESC,TXT_KEY_CIV_GERMANY_ADJECTIVE,...,None,8,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapGermany512.dds,TXT_KEY_CIV5_DAWN_GERMANY_TEXT,DOM_Bizmark.dds,AS2D_DOM_SPEECH_GERMANY,None,None
8,8,CIVILIZATION_GREECE,TXT_KEY_CIV_GREECE_DESC,None,TXT_KEY_CIV5_GREECE,None,1,1,TXT_KEY_CIV_GREECE_SHORT_DESC,TXT_KEY_CIV_GREECE_ADJECTIVE,...,None,9,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapMacedonia512.dds,TXT_KEY_CIV5_DAWN_GREECE_TEXT,DOM_Alex.dds,AS2D_DOM_SPEECH_GREECE,None,None
9,9,CIVILIZATION_INDIA,TXT_KEY_CIV_INDIA_DESC,None,TXT_KEY_CIV5_INDIA,None,1,1,TXT_KEY_CIV_INDIA_SHORT_DESC,TXT_KEY_CIV_INDIA_ADJECTIVE,...,None,11,CIV_COLOR_ATLAS,CIV_ALPHA_ATLAS,MapIndia512.dds,TXT_KEY_CIV5_DAWN_INDIA_TEXT,DOM_Gandhi.dds,AS2D_DOM_SPEECH_INDIA,None,None


In [17]:
pd.read_sql_query("SELECT * FROM Civilizations", cnx).columns

Index(['ID', 'Type', 'Description', 'Civilopedia', 'CivilopediaTag',
       'Strategy', 'Playable', 'AIPlayable', 'ShortDescription', 'Adjective',
       'DefaultPlayerColor', 'ArtDefineTag', 'ArtStyleType', 'ArtStyleSuffix',
       'ArtStylePrefix', 'DerivativeCiv', 'PortraitIndex', 'IconAtlas',
       'AlphaIconAtlas', 'MapImage', 'DawnOfManQuote', 'DawnOfManImage',
       'DawnOfManAudio', 'PackageID', 'SoundtrackTag'],
      dtype='object')

In [18]:
pd.read_sql_query("SELECT * FROM Civilizations", cnx)[['DefaultPlayerColor']]

,DefaultPlayerColor
0,PLAYERCOLOR_AMERICA
1,PLAYERCOLOR_ARABIA
2,PLAYERCOLOR_AZTEC
3,PLAYERCOLOR_CHINA
4,PLAYERCOLOR_EGYPT
5,PLAYERCOLOR_ENGLAND
6,PLAYERCOLOR_FRANCE
7,PLAYERCOLOR_GERMANY
8,PLAYERCOLOR_GREECE
9,PLAYERCOLOR_INDIA
